# SUP ML 3 - PREDICT

# Librerias

In [135]:
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action = 'ignore', category = FutureWarning)

# imports best practice pandas
import os

import numpy as np
import pandas as pd
import missingno as msno
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib
!pip install dill
import dill

#--------------------------------------------------------
# imports best practice sklearn
import sklearn
from sklearn.feature_selection import VarianceThreshold
from sklearn import set_config

from sklearn.tree import DecisionTreeClassifier

# preprocessing
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from scipy import stats

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer

# evaluacion
from sklearn.metrics import mean_absolute_error, r2_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import metrics

#interpretabilidad
!pip install shap
import shap

# modelos
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier

# pipelines
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
set_config(transform_output = "pandas")

# Carga modelo

* Cargar el modelo
* Obtener la lista de model features

In [136]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [137]:
CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

model = pickle.load(open(os.path.join(DATA_PATH, 'classifier.pkl'), 'rb'))

In [138]:
# model.feature_names_in_ para otros modelos

boost = model.booster_
features_model = boost.feature_name()
features_model

['oe_cat_1__categorical_impute_1__new_cell_N',
 'oe_cat_1__categorical_impute_1__new_cell_U',
 'oe_cat_1__categorical_impute_1__new_cell_Y',
 'oe_cat_1__categorical_impute_1__crclscod_A',
 'oe_cat_1__categorical_impute_1__crclscod_A2',
 'oe_cat_1__categorical_impute_1__crclscod_AA',
 'oe_cat_1__categorical_impute_1__crclscod_B',
 'oe_cat_1__categorical_impute_1__crclscod_BA',
 'oe_cat_1__categorical_impute_1__crclscod_C',
 'oe_cat_1__categorical_impute_1__crclscod_CA',
 'oe_cat_1__categorical_impute_1__crclscod_DA',
 'oe_cat_1__categorical_impute_1__crclscod_EA',
 'oe_cat_1__categorical_impute_1__crclscod_Ot',
 'oe_cat_1__categorical_impute_1__crclscod_U',
 'oe_cat_1__categorical_impute_1__crclscod_ZA',
 'oe_cat_1__categorical_impute_1__prizm_social_one_C',
 'oe_cat_1__categorical_impute_1__prizm_social_one_R',
 'oe_cat_1__categorical_impute_1__prizm_social_one_S',
 'oe_cat_1__categorical_impute_1__prizm_social_one_T',
 'oe_cat_1__categorical_impute_1__prizm_social_one_U',
 'oe_cat_1__

# Carga PREDICT dataset

In [139]:
CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/data")

X_pred = pd.read_csv(os.path.join(DATA_PATH, "telecom_churn_PREDICT.csv"))

In [140]:
X_pred.shape

(10000, 99)

In [141]:
X_pred.head()

,rev,mou,totmrc,da,ovrmou,ovrrev,vceovr,datovr,roam,change_mou,...,forgntvl,ethnic,kid0_2,kid3_5,kid6_10,kid11_15,kid16_17,creditcd,eqpdays,Customer_ID
0,30.8350,136.75,29.9900,0.2475,1.25,0.500,0.5,0.000,0.0975,48.25,...,1.0,U,U,U,U,U,U,Y,216.0,1090001
1,35.8475,352.75,24.2700,0.4950,23.25,9.285,8.7,0.585,1.8000,-352.75,...,0.0,N,U,U,U,U,U,N,101.0,1090002
2,30.3275,241.50,39.9900,0.0000,0.00,0.000,0.0,0.000,0.0000,-86.50,...,0.0,N,U,U,U,U,U,Y,262.0,1090003
3,154.6925,2297.00,149.9900,4.7025,0.00,0.000,0.0,0.000,0.0000,119.00,...,0.0,S,U,U,U,U,U,Y,127.0,1090004
4,156.0050,542.25,48.9475,0.7425,70.75,28.295,28.1,0.195,0.0000,195.75,...,0.0,N,U,U,U,U,U,Y,37.0,1090005


# ML Preprocessing

Se deben realizar las mismas transformaciones que se aplicaron en el preprocessing del train:
  * Eliminar mismas variables
  * Imputar mismos valores a nulos
  * Aplicar mismo encoding de categoricos
  * ...todo lo necesario para replicar las variables con las que el modelo se entrenó

NO SE DEBEN repetir:
  * Analisis visual de los datos
  * Analisis de correlaciones
  * Analisis de varianza
  * Calculo de metricas que varien con la distribución

In [142]:
# cargamos la lista de valores boolean a corregir

CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

with open(os.path.join(DATA_PATH, 'Lista_limpieza_valores_boolean.joblib'), 'rb') as io:
    Lista_limpieza_valores_boolean=dill.load(io)

In [143]:
Lista_limpieza_valores_boolean

['asl_flag', 'creditcd']

In [144]:
# cargamos la funcion de valores boolean a corregir

CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

with open(os.path.join(DATA_PATH, 'limpieza_valores_boolean.joblib'), 'rb') as io:
    limpieza_valores_boolean=dill.load(io)

In [145]:
# cargamos la función de negative_eqp_days_function

CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

with open(os.path.join(DATA_PATH, 'negative_eqp_days_function.joblib'), 'rb') as io:
    negative_eqp_days_function=dill.load(io)

In [146]:
negative_eqp_days_function

FunctionTransformer(func=<function imputar_negativos_eqp_days at 0x78e3702c9ab0>)

In [147]:
# cargamos la moda de la función de negative_eqp_days_function

CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

with open(os.path.join(DATA_PATH, 'moda.joblib'), 'rb') as io:
    moda=dill.load(io)

In [148]:
moda

310

In [149]:
# cargamos la lista de agrupación de dwllsize

CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

with open(os.path.join(DATA_PATH, 'values_to_replace_dwllsize.joblib'), 'rb') as io:
    values_to_replace_dwllsize=dill.load(io)

In [150]:
values_to_replace_dwllsize

['N', 'D', 'K', 'E', 'L', 'F', 'M', 'G', 'H', 'I']

In [151]:
# cargamos la lista de agrupación de ethnic

CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

with open(os.path.join(DATA_PATH, 'values_to_replace_ethnic.joblib'), 'rb') as io:
    values_to_replace_ethnic=dill.load(io)

In [152]:
values_to_replace_ethnic

['F', 'B', 'R', 'D', 'M', 'P', 'X']

In [153]:
# cargamos la lista de agrupación de crclscod

CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

with open(os.path.join(DATA_PATH, 'values_to_replace_crclscod.joblib'), 'rb') as io:
    values_to_replace_crclscod=dill.load(io)

In [154]:
print(values_to_replace_crclscod)

['E', 'E4', 'GA', 'D', 'G', 'I', 'JF', 'Z', 'J', 'M', 'C2', 'D4', 'Z4', 'K', 'W', 'V1', 'U1', 'EM', 'B2', 'Y', 'EC', 'O', 'CY', 'E2', 'CC', 'D5', 'C5', 'IF', 'ZY', 'Z1', 'Z5', 'H', 'TP', 'D2', 'GY', 'L', 'EF', 'Z2', 'A3', 'P1', 'S', 'V', 'ZF']


In [155]:
CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/model")

pipe = pickle.load(open(os.path.join(DATA_PATH, 'pipeline.pkl'), 'rb'))

In [156]:
pipe

Pipeline(steps=[('impute',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_impute',
                                                  SimpleImputer(fill_value=-999,
                                                                strategy='constant'),
                                                  ['rev', 'mou', 'totmrc', 'da',
                                                   'ovrmou', 'ovrrev', 'vceovr',
                                                   'datovr', 'roam',
                                                   'change_mou', 'change_rev',
                                                   'drop_vce', 'drop_dat',
                                                   'blck_vce', 'blck_dat',
                                                   'unan_vce', 'unan_dat',
                                                   'plcd_vce', 'plcd_dat',
                                                   'recv_vce', 'rec...
                                                                sparse_output=False),
                                                  ['categorical_impute_2__hnd_webcap']),
                                                 ('oe_cat_3',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['categorical_impute_3__area']),
                                                 ('oe_bool',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['boolean_impute__truck',
                                                   'boolean_impute__rv',
                                                   'boolean_impute__forgntvl',
                                                   'boolean_impute__creditcd',
                                                   'boolean_impute__asl_flag'])]))])

In [157]:
limpieza_valores_boolean(X_pred, Lista_limpieza_valores_boolean)

In [158]:
for i in Lista_limpieza_valores_boolean:

        print(X_pred[i].value_counts(dropna=False))
        print('')
        print('--------------')
        print('')
        print('')

print(X_pred[Lista_limpieza_valores_boolean].info())

0       6663
1       3337
<NA>       0
Name: asl_flag, dtype: Int64

--------------


1       5518
0       4201
<NA>     281
Name: creditcd, dtype: Int64

--------------


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   asl_flag  10000 non-null  Int64
 1   creditcd  9719 non-null   Int64
dtypes: Int64(2)
memory usage: 175.9 KB
None


In [159]:
len(X_pred[X_pred['eqpdays']<0])

31

In [160]:
X_pred['eqpdays'] = negative_eqp_days_function.transform(X_pred['eqpdays'])

In [161]:
len(X_pred[X_pred['eqpdays']<0])

0

In [162]:
X_pred['dwllsize'].value_counts(dropna=False)

NaN    4544
A      4087
B       462
J       150
C       140
N        95
O        93
D        69
K        66
L        63
E        53
G        46
F        37
H        37
M        31
I        27
Name: dwllsize, dtype: int64

In [163]:
# reemplazo para dwllsize

for i in values_to_replace_dwllsize:

    X_pred['dwllsize'] = np.where(X_pred['dwllsize'] == i, 'Ot', X_pred['dwllsize'])

X_pred['dwllsize'].value_counts(dropna=False)

NaN    4544
A      4087
Ot      524
B       462
J       150
C       140
O        93
Name: dwllsize, dtype: int64

In [164]:
X_pred['ethnic'].value_counts(dropna=False)

N      2989
H      1652
S      1123
U       968
G       458
Z       445
P       394
O       387
I       302
NaN     281
C       244
J       216
F       187
R        99
B        95
X        82
D        66
M        12
Name: ethnic, dtype: int64

In [165]:
# reemplazo para ethnic

for i in values_to_replace_ethnic:

    X_pred['ethnic'] = np.where(X_pred['ethnic'] == i, 'Ot', X_pred['ethnic'])

X_pred['ethnic'].value_counts(dropna=False)

N      2989
H      1652
S      1123
U       968
Ot      935
G       458
Z       445
O       387
I       302
NaN     281
C       244
J       216
Name: ethnic, dtype: int64

In [166]:
X_pred['crclscod'].value_counts(dropna=False)

AA    2357
BA    1397
A     1394
CA     945
EA     898
E4     528
DA     468
B      367
ZA     316
D4     194
CY     169
Z4     159
ZY     107
D5      98
A2      90
B2      67
C5      67
U       51
Z5      48
C       38
GY      29
C2      28
J       21
Y       16
M       16
EC      16
GA      16
W       13
K       13
I       12
EM      11
U1      10
E        9
CC       6
G        5
Z        4
P1       4
D        3
JF       2
E2       2
L        2
H        1
Z1       1
Z2       1
O        1
Name: crclscod, dtype: int64

In [167]:
# reemplazo para crclscod

for i in values_to_replace_crclscod:

    X_pred['crclscod'] = np.where(X_pred['crclscod'] == i, 'Ot', X_pred['crclscod'])

X_pred['crclscod'].value_counts(dropna=False)

AA    2357
Ot    1679
BA    1397
A     1394
CA     945
EA     898
DA     468
B      367
ZA     316
A2      90
U       51
C       38
Name: crclscod, dtype: int64

In [168]:
X_pred.set_index('Customer_ID', drop=True, inplace=True)

In [169]:
X_pred.head()

,rev,mou,totmrc,da,ovrmou,ovrrev,vceovr,datovr,roam,change_mou,...,dwllsize,forgntvl,ethnic,kid0_2,kid3_5,kid6_10,kid11_15,kid16_17,creditcd,eqpdays
Customer_ID,,,,,,,,,,,,,,,,,,,,,
1090001,30.8350,136.75,29.9900,0.2475,1.25,0.500,0.5,0.000,0.0975,48.25,...,NaN,1.0,U,U,U,U,U,U,1,216.0
1090002,35.8475,352.75,24.2700,0.4950,23.25,9.285,8.7,0.585,1.8000,-352.75,...,NaN,0.0,N,U,U,U,U,U,0,101.0
1090003,30.3275,241.50,39.9900,0.0000,0.00,0.000,0.0,0.000,0.0000,-86.50,...,A,0.0,N,U,U,U,U,U,1,262.0
1090004,154.6925,2297.00,149.9900,4.7025,0.00,0.000,0.0,0.000,0.0000,119.00,...,A,0.0,S,U,U,U,U,U,1,127.0
1090005,156.0050,542.25,48.9475,0.7425,70.75,28.295,28.1,0.195,0.0000,195.75,...,NaN,0.0,N,U,U,U,U,U,1,37.0


In [170]:
X_pred.drop('infobase', axis='columns', inplace=True)

In [171]:
# no figure el Customer_ID

X_pred.columns

Index(['rev', 'mou', 'totmrc', 'da', 'ovrmou', 'ovrrev', 'vceovr', 'datovr',
       'roam', 'change_mou', 'change_rev', 'drop_vce', 'drop_dat', 'blck_vce',
       'blck_dat', 'unan_vce', 'unan_dat', 'plcd_vce', 'plcd_dat', 'recv_vce',
       'recv_sms', 'comp_vce', 'comp_dat', 'custcare', 'ccrndmou', 'cc_mou',
       'inonemin', 'threeway', 'mou_cvce', 'mou_cdat', 'mou_rvce',
       'owylis_vce', 'mouowylisv', 'iwylis_vce', 'mouiwylisv', 'peak_vce',
       'peak_dat', 'mou_peav', 'mou_pead', 'opk_vce', 'opk_dat', 'mou_opkv',
       'mou_opkd', 'drop_blk', 'attempt', 'complete', 'callfwdv', 'callwait',
       'months', 'uniqsubs', 'actvsubs', 'new_cell', 'crclscod', 'asl_flag',
       'totcalls', 'totmou', 'totrev', 'adjrev', 'adjmou', 'adjqty', 'avgrev',
       'avgmou', 'avgqty', 'avg3mou', 'avg3qty', 'avg3rev', 'avg6mou',
       'avg6qty', 'avg6rev', 'prizm_social_one', 'area', 'dualband',
       'refurb_new', 'hnd_price', 'phones', 'models', 'hnd_webcap', 'truck',
       'rv', 'ownr

In [172]:
target = 'remainder__remainder__churn'

X_pred['churn'] = 0

X_pred = pipe.transform(X_pred)

X_pred.drop(target, axis='columns', inplace=True)

In [173]:
# guardamos un diccionario con todas las imputaciones

numeric_impute =  dict(zip(pipe.steps[0][1].transformers_[0][2] , pipe.steps[0][1].transformers_[0][1].statistics_))
categorical_impute_1 =  dict(zip(pipe.steps[0][1].transformers_[1][2] , pipe.steps[0][1].transformers_[1][1].statistics_))
categorical_impute_2 =  dict(zip(pipe.steps[0][1].transformers_[2][2] , pipe.steps[0][1].transformers_[2][1].statistics_))
categorical_impute_3 =  dict(zip(pipe.steps[0][1].transformers_[3][2] , pipe.steps[0][1].transformers_[3][1].statistics_))
boolean_impute =  dict(zip(pipe.steps[0][1].transformers_[4][2] , pipe.steps[0][1].transformers_[4][1].statistics_))

print('Se han imputado los siguientes valores en las siguientes columnas para los nulos:')
print('')
print('numeric_impute', numeric_impute)
print('categorical_impute_1', categorical_impute_1)
print('categorical_impute_2', categorical_impute_2)
print('categorical_impute_3', categorical_impute_3)
print('boolean_impute', boolean_impute)


Se han imputado los siguientes valores en las siguientes columnas para los nulos:

numeric_impute {'rev': -999.0, 'mou': -999.0, 'totmrc': -999.0, 'da': -999.0, 'ovrmou': -999.0, 'ovrrev': -999.0, 'vceovr': -999.0, 'datovr': -999.0, 'roam': -999.0, 'change_mou': -999.0, 'change_rev': -999.0, 'drop_vce': -999.0, 'drop_dat': -999.0, 'blck_vce': -999.0, 'blck_dat': -999.0, 'unan_vce': -999.0, 'unan_dat': -999.0, 'plcd_vce': -999.0, 'plcd_dat': -999.0, 'recv_vce': -999.0, 'recv_sms': -999.0, 'comp_vce': -999.0, 'comp_dat': -999.0, 'custcare': -999.0, 'ccrndmou': -999.0, 'cc_mou': -999.0, 'inonemin': -999.0, 'threeway': -999.0, 'mou_cvce': -999.0, 'mou_cdat': -999.0, 'mou_rvce': -999.0, 'owylis_vce': -999.0, 'mouowylisv': -999.0, 'iwylis_vce': -999.0, 'mouiwylisv': -999.0, 'peak_vce': -999.0, 'peak_dat': -999.0, 'mou_peav': -999.0, 'mou_pead': -999.0, 'opk_vce': -999.0, 'opk_dat': -999.0, 'mou_opkv': -999.0, 'mou_opkd': -999.0, 'drop_blk': -999.0, 'attempt': -999.0, 'complete': -999.0, 'cal

In [174]:
# mostramos los pasos aplicados del OneHotEncoder

pipe[1]

ColumnTransformer(remainder='passthrough',
                  transformers=[('oe_cat_1',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['categorical_impute_1__new_cell',
                                  'categorical_impute_1__crclscod',
                                  'categorical_impute_1__prizm_social_one',
                                  'categorical_impute_1__dualband',
                                  'categorical_impute_1__refurb_new',
                                  'categorical_impute_1__ownrent',
                                  'categorical_im...
                                               sparse_output=False),
                                 ['categorical_impute_2__hnd_webcap']),
                                ('oe_cat_3',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['categorical_impute_3__area']),
                                ('oe_bool',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['boolean_impute__truck', 'boolean_impute__rv',
                                  'boolean_impute__forgntvl',
                                  'boolean_impute__creditcd',
                                  'boolean_impute__asl_flag'])])

In [175]:
X_pred.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10000 entries, 1090001 to 1100000
Data columns (total 185 columns):
 #    Column                                                              Non-Null Count  Dtype  
---   ------                                                              --------------  -----  
 0    oe_cat_1__categorical_impute_1__new_cell_N                          10000 non-null  float64
 1    oe_cat_1__categorical_impute_1__new_cell_U                          10000 non-null  float64
 2    oe_cat_1__categorical_impute_1__new_cell_Y                          10000 non-null  float64
 3    oe_cat_1__categorical_impute_1__crclscod_A                          10000 non-null  float64
 4    oe_cat_1__categorical_impute_1__crclscod_A2                         10000 non-null  float64
 5    oe_cat_1__categorical_impute_1__crclscod_AA                         10000 non-null  float64
 6    oe_cat_1__categorical_impute_1__crclscod_B                          10000 non-null  float64


# Check model features

* Comprobar que tenemos en el dataset preprocesado todas las model features, de lo contrario no podremos hacer predict.
* Ordenar las variables en mismo orden que las model features

In [176]:
features_test = list(X_pred.columns)

print('Columnas en dataset:',len(features_test))
print('Variables en modelo:',len(features_model))
print('¿Match?:', features_model == features_test)

Columnas en dataset: 185
Variables en modelo: 157
¿Match?: False


In [177]:
missing_features = [i for i in features_model if i not in features_test]
print('Variables que faltan el el dataset:\n', missing_features)

Variables que faltan el el dataset:
 ['oe_cat_3__categorical_impute_3__area_ATLANTIC_SOUTH_AREA', 'oe_cat_3__categorical_impute_3__area_CALIFORNIA_NORTH_AREA', 'oe_cat_3__categorical_impute_3__area_CENTRAL/SOUTH_TEXAS_AREA', 'oe_cat_3__categorical_impute_3__area_CHICAGO_AREA', 'oe_cat_3__categorical_impute_3__area_DALLAS_AREA', 'oe_cat_3__categorical_impute_3__area_DC/MARYLAND/VIRGINIA_AREA', 'oe_cat_3__categorical_impute_3__area_GREAT_LAKES_AREA', 'oe_cat_3__categorical_impute_3__area_HOUSTON_AREA', 'oe_cat_3__categorical_impute_3__area_LOS_ANGELES_AREA', 'oe_cat_3__categorical_impute_3__area_MIDWEST_AREA', 'oe_cat_3__categorical_impute_3__area_NEW_ENGLAND_AREA', 'oe_cat_3__categorical_impute_3__area_NEW_YORK_CITY_AREA', 'oe_cat_3__categorical_impute_3__area_NORTH_FLORIDA_AREA', 'oe_cat_3__categorical_impute_3__area_NORTHWEST/ROCKY_MOUNTAIN_AREA', 'oe_cat_3__categorical_impute_3__area_OHIO_AREA', 'oe_cat_3__categorical_impute_3__area_PHILADELPHIA_AREA', 'oe_cat_3__categorical_impute_3

In [178]:
# agregamos columnas faltantes

for col in missing_features:
  X_pred[col]= 0

In [179]:
drop_features = [i for i in features_test if i not in features_model]
print('Variables que debes eliminar de tu dataset:\n', drop_features)

Variables que debes eliminar de tu dataset:
 ['oe_cat_1__categorical_impute_1__refurb_new_N', 'oe_cat_1__categorical_impute_1__refurb_new_U', 'oe_cat_1__categorical_impute_1__ownrent_U', 'oe_cat_1__categorical_impute_1__kid0_2_U', 'oe_cat_1__categorical_impute_1__kid3_5_U', 'oe_cat_1__categorical_impute_1__kid6_10_U', 'oe_cat_1__categorical_impute_1__kid11_15_U', 'oe_cat_1__categorical_impute_1__kid16_17_U', 'oe_cat_3__categorical_impute_3__area_ATLANTIC SOUTH AREA', 'oe_cat_3__categorical_impute_3__area_CALIFORNIA NORTH AREA', 'oe_cat_3__categorical_impute_3__area_CENTRAL/SOUTH TEXAS AREA', 'oe_cat_3__categorical_impute_3__area_CHICAGO AREA', 'oe_cat_3__categorical_impute_3__area_DALLAS AREA', 'oe_cat_3__categorical_impute_3__area_DC/MARYLAND/VIRGINIA AREA', 'oe_cat_3__categorical_impute_3__area_GREAT LAKES AREA', 'oe_cat_3__categorical_impute_3__area_HOUSTON AREA', 'oe_cat_3__categorical_impute_3__area_LOS ANGELES AREA', 'oe_cat_3__categorical_impute_3__area_MIDWEST AREA', 'oe_cat_3_

In [180]:
X_pred.drop(columns = drop_features, inplace=True)

In [181]:
features_test = list(X_pred.columns)

print('Columnas en dataset:',len(features_test))
print('Variables en modelos:',len(features_model))
print('¿Match?:', features_model == features_test)

Columnas en dataset: 157
Variables en modelos: 157
¿Match?: False


In [182]:
# Reordena variables

X_pred = X_pred[features_model]

In [183]:
features_test = list(X_pred.columns)

print('Columnas en dataset:',len(features_test))
print('Variables en modelos:',len(features_model))
print('¿Match?:', features_model == features_test)

Columnas en dataset: 157
Variables en modelos: 157
¿Match?: True


# Rescaling

* Si se entrenó el modelo con un dataset estandarizado, estandarizar con mismo scaler.

In [184]:
# no hace falta estandarizar

# PREDICT

* predict() y predict_proba()

In [185]:
predictions = model.predict(X_pred)

In [186]:
# prediccion de churn en mes + 1

predictions

array([0, 0, 0, ..., 1, 1, 1])

In [187]:
predict_proba = model.predict_proba(X_pred)

In [188]:
# probabilidad de no churn mes + 1 (izquierda) y churn en mes + 1 (derecha)

predict_proba

array([[0.58195155, 0.41804845],
       [0.5820598 , 0.4179402 ],
       [0.80719445, 0.19280555],
       ...,
       [0.02646808, 0.97353192],
       [0.07580743, 0.92419257],
       [0.16617402, 0.83382598]])

In [189]:
# probabilidad de churn en mes + 1

predict_proba[:,1]

array([0.41804845, 0.4179402 , 0.19280555, ..., 0.97353192, 0.92419257,
       0.83382598])

In [190]:
X_pred.head()

,oe_cat_1__categorical_impute_1__new_cell_N,oe_cat_1__categorical_impute_1__new_cell_U,oe_cat_1__categorical_impute_1__new_cell_Y,oe_cat_1__categorical_impute_1__crclscod_A,oe_cat_1__categorical_impute_1__crclscod_A2,oe_cat_1__categorical_impute_1__crclscod_AA,oe_cat_1__categorical_impute_1__crclscod_B,oe_cat_1__categorical_impute_1__crclscod_BA,oe_cat_1__categorical_impute_1__crclscod_C,oe_cat_1__categorical_impute_1__crclscod_CA,...,remainder__numeric_impute__avg6mou,remainder__numeric_impute__avg6qty,remainder__numeric_impute__avg6rev,remainder__numeric_impute__hnd_price,remainder__numeric_impute__phones,remainder__numeric_impute__lor,remainder__numeric_impute__adults,remainder__numeric_impute__income,remainder__numeric_impute__numbcars,remainder__numeric_impute__eqpdays
Customer_ID,,,,,,,,,,,,,,,,,,,,,
1090001,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,160.0,66.0,31.0,149.98999,1.0,-999.0,1.0,3.0,-999.0,216.0
1090002,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,666.0,272.0,51.0,99.98999,2.0,-999.0,-999.0,-999.0,-999.0,101.0
1090003,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,273.0,109.0,31.0,129.98999,1.0,13.0,3.0,6.0,1.0,262.0
1090004,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,2452.0,1171.0,278.0,79.98999,2.0,3.0,3.0,6.0,1.0,127.0
1090005,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,349.0,103.0,91.0,149.98999,2.0,1.0,1.0,6.0,1.0,37.0


In [191]:
X_pred.index

Int64Index([1090001, 1090002, 1090003, 1090004, 1090005, 1090006, 1090007,
            1090008, 1090009, 1090010,
            ...
            1099991, 1099992, 1099993, 1099994, 1099995, 1099996, 1099997,
            1099998, 1099999, 1100000],
           dtype='int64', name='Customer_ID', length=10000)

In [192]:
df_telecom_final_with_predictions = pd.DataFrame()

In [193]:
df_telecom_final_with_predictions['Customer_ID'] = X_pred.index

df_telecom_final_with_predictions['PREDICT'] = predictions

df_telecom_final_with_predictions['PREDICT_PROBA'] = predict_proba[:,1]

In [194]:
df_telecom_final_with_predictions

,Customer_ID,PREDICT,PREDICT_PROBA
0,1090001,0,0.418048
1,1090002,0,0.417940
2,1090003,0,0.192806
3,1090004,1,0.557188
4,1090005,0,0.275847
...,...,...,...
9995,1099996,1,0.968464
9996,1099997,1,0.916188
9997,1099998,1,0.973532
9998,1099999,1,0.924193


# Guarda predicciones

* Guardar las predicciones en data path. Cada fila debe estar etiquetada con el ID.

In [195]:
CWD = os.getcwd()

DATA_PATH = os.path.join(CWD, "/content/drive/MyDrive/EJERCICIO-ML-Sup/data")

df_telecom_final_with_predictions.to_csv(os.path.join(DATA_PATH, 'df_telecom_final_with_predictions.csv'))